# Alternating least squares for Tucker model

```
Copyright 2025 National Technology & Engineering Solutions of Sandia,
LLC (NTESS). Under the terms of Contract DE-NA0003525 with NTESS, the
U.S. Government retains certain rights in this software.
```

The function `pyttb.tucker_als()` computes the best rank-$(R_1,R_2,\ldots,R_n)$ approximation of tensor $\mathcal{X}$, according to the specified dimensions in the vector $(R_1,R_2,\ldots,R_n)$. The form of $\mathcal{X}$ can be a `tensor`, `sptensor`, `ktensor`, or `ttensor`. The result is a `ttensor`.

The method is originally from Tucker (1966) and later revisited in De Lathauwer et al. (2000).

* L. R. Tucker, Some mathematical notes on three-mode factor analysis, Psychometrika, 31:279-311, 1966, http://dx.doi.org/10.1007/BF02289464
* L. De Lathauwer, B. De Moor, J. Vandewalle, On the best rank-1 and rank-(R_1, R_2, R_N) approximation of higher-order tensors, SIAM J. Matrix Analysis and Applications, 21:1324-1342, 2000, http://doi.org/10.1137/S0895479898346995

Note: Oftentimes it's better to use `pyttb.hosvd()` instead.

In [ ]:
from __future__ import annotations

import os
import sys

import numpy as np

import pyttb as ttb

## Problem setup

In [ ]:
np.random.seed(0)  # Set seed for reproducibility
X = ttb.sptenrand(
    [5, 4, 3], nonzeros=10
)  # Create a tensor with 10 nonzeros using the 'nonzeros' param.
X

## Basic usage

### Create an approximation with all ranks equal to 2

In [ ]:
T = ttb.tucker_als(X, 2)  # best rank(2,2,2) approximation
T

## Main options

### Create an approximation with specific ranks of [2, 2, 1]

In [ ]:
T = ttb.tucker_als(X, [2, 2, 1])  # best rank(2,2,1) approximation
T

### Use a different ordering of the dimensions

In [ ]:
T = ttb.tucker_als(X, 2, dimorder=[2, 1, 0])
T

### Use the `"nvecs"` initialization method
This initialization is more expensive but generally works very well.

In [ ]:
T = ttb.tucker_als(X, 2, dimorder=[0, 1, 2], init="nvecs")
T

### Specify the initial guess manually

In [ ]:
U0 = [np.random.rand(5, 2), np.random.rand(4, 2), np.random.rand(3, 2)]
T = ttb.tucker_als(X, 2, dimorder=[0, 1, 2], init=U0)
T

## Interpreting the output

`tucker_als` returns the Tucker model `M` as a `ttensor`, the initial guess `Minit`, and a dictionary `info` with the number of iterations (`iters`), the norm of the residual (`normresidual`), the fit (`fit`, the fraction of the data explained by the model), and the parameters used (`params`).

In [ ]:
M, Minit, info = ttb.tucker_als(X, [2, 2, 1], printitn=0)
print(f"Core shape: {M.core.shape}")
print(f"Factor matrix shapes: {[f.shape for f in M.factor_matrices]}")
print(
    f"Iterations: {info['iters']}, "
    f"residual norm: {info['normresidual']:.3f}, "
    f"fit: {info['fit']:.3f}"
)

## Worked example

Build a noisy tensor with exact Tucker rank `(2, 2, 1)`, recover the model with `tucker_als`, and verify that the relative error is on the order of the noise.

In [ ]:
np.random.seed(0)
G = ttb.tenrand((2, 2, 1))
U = [np.random.rand(s, r) for s, r in zip((8, 6, 4), (2, 2, 1))]
T_true = ttb.ttensor(G, U)
noise = ttb.tenrand((8, 6, 4))
X_noisy = T_true.full() + 1e-3 * T_true.norm() * noise / noise.norm()

T_fit, _, info_fit = ttb.tucker_als(X_noisy, [2, 2, 1], init="nvecs", printitn=0)
rel_err = (X_noisy - T_fit.full()).norm() / X_noisy.norm()
print(f"Relative error: {rel_err:.1e}")